# STDL-Net 月球线性构造多类别分割训练

## 使用前
1. 已上传 Dataset `lunar-data`（含代码 + 数据 + 预训练权重）
2. 在 Notebook 右侧 **Add Data** 添加 `lunar-data`
3. 选择 **GPU T4 x2** 加速器
4. 运行所有单元格

In [ ]:
# Cell 1: 安装依赖
!pip install rasterio segmentation-models-pytorch -q

In [ ]:
# Cell 2: 检查 GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    total_gb = getattr(props, 'total_memory', 0) or getattr(props, 'total_mem', 0)
    print(f'显存: {total_gb / 1024**3:.1f} GB')

In [ ]:
# Cell 3: 检查数据路径
import os

# v3: 去重叠版本
DATA_ROOT = '/kaggle/input/lunar-data-v3/dataset/dataset'
CODE_ROOT = '/kaggle/input/lunar-data-v3/STDL-Net/STDL-Net'

# 兼容: 如果 v3 不存在则回退到 v2
if not os.path.isdir(DATA_ROOT):
    DATA_ROOT = '/kaggle/input/lunar-data-v2/dataset/dataset'
    CODE_ROOT = '/kaggle/input/lunar-data-v2/STDL-Net/STDL-Net'
    print('WARNING: lunar-data-v3 not found, falling back to v2')

# 列出目录确认
print(f'DATA_ROOT: {DATA_ROOT}')
print('=== 数据目录 ===')
for d in os.listdir(DATA_ROOT):
    full = os.path.join(DATA_ROOT, d)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f'  {d}/ ({n} items)')

print('\n=== 代码目录 ===')
for f in os.listdir(CODE_ROOT):
    print(f'  {f}')

In [ ]:
# Cell 4: 设置路径
TRAIN_IMG  = os.path.join(DATA_ROOT, 'train', 'image')
TRAIN_MASK = os.path.join(DATA_ROOT, 'train', 'mask')
TEST_IMG   = os.path.join(DATA_ROOT, 'test', 'image')
TEST_MASK  = os.path.join(DATA_ROOT, 'test', 'mask')
PRETRAIN   = os.path.join(DATA_ROOT, 'pretrain')

# 验证
for p, name in [(TRAIN_IMG, 'train/image'), (TRAIN_MASK, 'train/mask'),
                 (TEST_IMG, 'test/image'), (TEST_MASK, 'test/mask'),
                 (PRETRAIN, 'pretrain')]:
    if os.path.isdir(p):
        print(f'OK  {name}: {len(os.listdir(p))} files')
    else:
        print(f'MISSING  {name}: {p}')

In [ ]:
# Cell 5: 复制代码到工作目录 (Kaggle input 只读, 需要复制才能修改)
import shutil

CODE_DST = '/kaggle/working/code'
os.makedirs(CODE_DST, exist_ok=True)

for f in os.listdir(CODE_ROOT):
    if f.endswith('.py'):
        shutil.copy2(os.path.join(CODE_ROOT, f), os.path.join(CODE_DST, f))
        print(f'copied: {f}')

import sys
sys.path.insert(0, CODE_DST)
print(f'\nCode path added: {CODE_DST}')

In [ ]:
# Cell 6: 修改预训练权重路径并导入模型
import sys, traceback, re

swin_path = os.path.join(CODE_DST, '../models/swinv2unet.py')

# ---- 只替换 _SWIN_V2_PRETRAINED 字典里的路径 ----
with open(swin_path, 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(
    r"_SWIN_V2_PRETRAINED\s*=\s*\{[^}]+\}",
    f"""_SWIN_V2_PRETRAINED = {{
    'tiny':  '{PRETRAIN}/swinv2_tiny_patch4_window16_256.pth',
    'small': '{PRETRAIN}/swinv2_small_patch4_window16_256.pth',
    'base':  '{PRETRAIN}/swinv2_base_patch4_window12to16_192to256_22kto1k_ft.pth',
}}""",
    content
)

with open(swin_path, 'w', encoding='utf-8') as f:
    f.write(content)

# ---- 验证 ----
print('=== 预训练路径验证 ===')
with open(swin_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        if '_SWIN_V2_PRETRAINED' in line or ('swinv2_' in line and '.pth' in line):
            print(f'  L{i}: {line.rstrip()[:100]}')

with open(swin_path, 'r', encoding='utf-8') as f:
    src = f.read()
if 'class Swin_LCSRB_DeformablePSP_FPNPAN' in src:
    print('\n✓ Swin_LCSRB_DeformablePSP_FPNPAN 类存在')
else:
    print('\n✗ ERROR: 类定义不在文件中!')

if 'class CoordinateAttention' in src:
    print('✓ CoordinateAttention 类存在')
else:
    print('✗ WARNING: CoordinateAttention 类不存在, 请检查上传的 swinv2unet.py')

# ---- 强制重新导入 (清除缓存) ----
for mod_name in list(sys.modules.keys()):
    if 'swinv2unet' in mod_name or 'MyDataset' in mod_name or 'metrics' in mod_name:
        del sys.modules[mod_name]

try:
    from swinv2unet import Swin_LCSRB_DeformablePSP_FPNPAN
    from MyDataset import MyDataset
    import metrics
    print('\n✓ 模型导入成功!')
except Exception as e:
    print(f'\n✗ 导入失败: {type(e).__name__}: {e}')
    traceback.print_exc()

In [ ]:
# Cell 7: 超参数配置 (R22: R17 baseline + DEM-Guided Fusion)

# ========== 核心参数 ==========
NUM_CLASSES   = 5
IN_CHANNELS   = 5
MODEL_SIZE    = 'small'
FREEZE_STAGES = 1
NUM_EPOCHS    = 60
MAX_STEPS     = 0
BATCH_SIZE    = 4
ACCUM_STEPS   = 1            # 等效 BS=4
LR            = 5e-5

# 数据增强
USE_AUGMENT   = True
USE_COPYPASTE = True
COPYPASTE_P   = 0.5

# ===== 模块开关 =====
USE_STRIP_POOLING   = False   # 不用
USE_COORD_ATTENTION = False   # 不用
USE_BOUNDARY_LOSS   = False   # R21 验证无效, 关闭
USE_DEM_GUIDED      = True    # R22: DEM-guided fusion (核心创新)
TERRAIN_CHANNELS    = 4       # 后4个通道: DEM, Slope, TPI, Curvature

# Early stopping
EARLY_STOP    = True
PATIENCE      = 12

# 导出开关
SAVE_ALL_TEST_ON_BEST = True
SAVE_PRED_MASK_PNG    = True
SAVE_PRED_VIS_PNG     = True

RESULT_DIR = '/kaggle/working/result'
os.makedirs(RESULT_DIR, exist_ok=True)

print(f'Model: {MODEL_SIZE}, Epochs: {NUM_EPOCHS}, BS: {BATCH_SIZE}x{ACCUM_STEPS}(eff={BATCH_SIZE*ACCUM_STEPS}), LR: {LR}')
print(f'Freeze stages: {FREEZE_STAGES}')
print(f'Augmentation: {USE_AUGMENT}, CopyPaste: {USE_COPYPASTE} (p={COPYPASTE_P})')
print(f'StripPooling: {USE_STRIP_POOLING}, CoordAttention: {USE_COORD_ATTENTION}')
print(f'BoundaryLoss: {USE_BOUNDARY_LOSS}, DEM-Guided: {USE_DEM_GUIDED} (terrain_channels={TERRAIN_CHANNELS})')
print(f'EarlyStop: {EARLY_STOP} (patience={PATIENCE})')

In [ ]:
# Cell 8: 创建模型 (兼容新旧版 swinv2unet.py)
import inspect

device = torch.device('cuda:0')

# 检查模型构造函数支持哪些参数 (兼容旧版本)
init_sig = inspect.signature(Swin_LCSRB_DeformablePSP_FPNPAN.__init__)
init_params = set(init_sig.parameters.keys())

model_kwargs = dict(
    size=MODEL_SIZE,
    num_classes=NUM_CLASSES,
    in_channels=IN_CHANNELS,
    pretrained=True,
)

# 只在参数受支持且 flag=True 时才传入
if 'use_strip_pooling' in init_params and USE_STRIP_POOLING:
    model_kwargs['use_strip_pooling'] = True
if 'use_coord_attention' in init_params and USE_COORD_ATTENTION:
    model_kwargs['use_coord_attention'] = True
if 'use_dem_guided' in init_params and USE_DEM_GUIDED:
    model_kwargs['use_dem_guided'] = True
    model_kwargs['terrain_channels'] = TERRAIN_CHANNELS

# 校验: R22 必须使用支持 DEM-guided 的 swinv2unet.py
if USE_DEM_GUIDED and 'use_dem_guided' not in init_params:
    raise RuntimeError(
        '当前 swinv2unet.py 不支持 use_dem_guided 参数!\n'
        '请重新上传最新版本的 swinv2unet.py 到 Kaggle Dataset.\n'
        f'当前可用参数: {sorted(init_params - {"self"})}'
    )

print(f'Model kwargs: {list(model_kwargs.keys())}')
print(f'Available init params: {sorted(init_params - {"self"})}')

model = Swin_LCSRB_DeformablePSP_FPNPAN(**model_kwargs).to(device)

if FREEZE_STAGES > 0:
    model.freeze_backbone_stages(FREEZE_STAGES)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Total params: {n_params:.1f}M, Trainable: {n_train:.1f}M')

In [ ]:
# Cell 9: 数据集 & DataLoader (R16: R13增强 + CopyPaste)
from torch.utils.data import DataLoader
import random

class AugmentedDataset(torch.utils.data.Dataset):
    """强化增强: 翻转 + 旋转 + 高斯噪声 + 亮度扰动 + Cutout + CopyPaste"""
    def __init__(self, base_dataset, copypaste=False, copypaste_p=0.5):
        self.base = base_dataset
        self.copypaste = copypaste
        self.copypaste_p = copypaste_p

        # 预索引含有少数类 (Fault=3, Graben=4) 的样本
        if self.copypaste:
            self.rare_indices = []
            print('CopyPaste: 预索引少数类样本...')
            for i in range(len(self.base)):
                _, mask, _ = self.base[i]
                classes_present = set(mask.unique().tolist())
                if 3 in classes_present or 4 in classes_present:
                    self.rare_indices.append(i)
            print(f'CopyPaste: 找到 {len(self.rare_indices)} 张含 Fault/Graben 的样本')

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask, name = self.base[idx]

        # --- CopyPaste: 从含少数类的样本中粘贴前景到当前图 ---
        if self.copypaste and self.rare_indices and random.random() < self.copypaste_p:
            donor_idx = random.choice(self.rare_indices)
            donor_img, donor_mask, _ = self.base[donor_idx]

            # 提取 donor 的少数类前景 (Fault=3 或 Graben=4)
            rare_fg = (donor_mask == 3) | (donor_mask == 4)
            if rare_fg.any():
                # 扩展 mask 到 (C, H, W) 用于像素替换
                fg_mask = rare_fg.unsqueeze(0).expand_as(img)
                img = torch.where(fg_mask, donor_img, img)
                mask = torch.where(rare_fg, donor_mask, mask)

        # --- 几何增强 (image + mask 同步) ---
        # 随机水平翻转
        if random.random() > 0.5:
            img = img.flip(-1)
            mask = mask.flip(-1)
        # 随机垂直翻转
        if random.random() > 0.5:
            img = img.flip(-2)
            mask = mask.flip(-2)
        # 随机 90° 旋转
        k = random.randint(0, 3)
        if k > 0:
            img = torch.rot90(img, k, [-2, -1])
            mask = torch.rot90(mask, k, [-2, -1])

        # --- 像素增强 (仅 image) ---
        # 高斯噪声 (p=0.5, std=0.02)
        if random.random() > 0.5:
            noise = torch.randn_like(img) * 0.02
            img = img + noise

        # 逐通道亮度偏移 (p=0.5, 范围 ±0.05)
        if random.random() > 0.5:
            C = img.shape[0]
            shift = (torch.rand(C, 1, 1) - 0.5) * 0.1  # [-0.05, +0.05]
            img = img + shift

        # 随机遮挡 Cutout (p=0.3, 1~3 个方块, 每块 32~64px)
        if random.random() > 0.7:
            _, H, W = img.shape
            n_holes = random.randint(1, 3)
            for _ in range(n_holes):
                size = random.randint(32, 64)
                y = random.randint(0, H - size)
                x = random.randint(0, W - size)
                img[:, y:y+size, x:x+size] = 0.0

        return img, mask, name

train_data_raw = MyDataset(images_dir=TRAIN_IMG, masks_dir=TRAIN_MASK)
test_data  = MyDataset(images_dir=TEST_IMG,  masks_dir=TEST_MASK)

if USE_AUGMENT:
    train_data = AugmentedDataset(
        train_data_raw,
        copypaste=USE_COPYPASTE,
        copypaste_p=COPYPASTE_P,
    )
    aug_str = '翻转 + 旋转 + 高斯噪声 + 亮度扰动 + Cutout'
    if USE_COPYPASTE:
        aug_str += f' + CopyPaste(p={COPYPASTE_P})'
    print(f'数据增强: {aug_str}')
else:
    train_data = train_data_raw

train_iter = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
test_iter  = DataLoader(test_data,  batch_size=1,          shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'训练集: {len(train_data)} 张, 测试集: {len(test_data)} 张')
print(f'训练 batches/epoch: {len(train_iter)}')

In [ ]:
# Cell 10: 损失函数 (R22: CE + Dice, 与 R17 一致) + 优化器
import torch.nn as nn
from torch import optim

# 类别权重: [背景, 皱脊, 月溪, 断层, 地堑]
class_weights = torch.tensor([0.15, 1.0, 1.3, 1.8, 2.5], dtype=torch.float32).to(device)
ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights)


def dice_loss(logits, targets, smooth=1.0):
    """前景类 Dice Loss"""
    probs = torch.softmax(logits, dim=1)
    dice = 0.0
    for c in range(1, logits.shape[1]):
        p = probs[:, c]
        g = (targets == c).float()
        inter = (p * g).sum(dim=(1, 2))
        union = p.sum(dim=(1, 2)) + g.sum(dim=(1, 2))
        dice += (1 - (2 * inter + smooth) / (union + smooth)).mean()
    return dice / (logits.shape[1] - 1)


def combined_loss(logits, targets):
    l_ce = ce_loss_fn(logits, targets)
    l_dice = dice_loss(logits, targets)
    return l_ce + 0.5 * l_dice

print(f'Loss: CE(weights={class_weights.cpu().tolist()}) + 0.5*Dice')

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

In [ ]:
# Cell 11: 训练循环 + early stopping + 全量导出
import os, csv, json, shutil
import numpy as np
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

if 'SAVE_ALL_TEST_ON_BEST' not in dir():
    SAVE_ALL_TEST_ON_BEST = True
if 'SAVE_PRED_MASK_PNG' not in dir():
    SAVE_PRED_MASK_PNG = True
if 'SAVE_PRED_VIS_PNG' not in dir():
    SAVE_PRED_VIS_PNG = True
if 'EARLY_STOP' not in dir():
    EARLY_STOP = True
if 'PATIENCE' not in dir():
    PATIENCE = 12

CLASS_NAMES = ['背景', '皱脊', '月溪', '断层', '地堑']
CLASS_COLORS = np.array([
    [0, 0, 0],
    [255, 0, 0],
    [0, 100, 255],
    [0, 200, 0],
    [255, 255, 0],
], dtype=np.uint8)
CLASS_COLORS_PLT = ['black', 'red', 'dodgerblue', 'green', 'orange']

from MyDataset import CHANNEL_MEAN, CHANNEL_STD

scaler = torch.amp.GradScaler('cuda')
best_miou = 0.0
best_export_dir = None
no_improve_count = 0

history = {
    'epoch': [],
    'train_loss': [], 'train_miou': [], 'train_acc': [],
    'test_loss':  [], 'test_miou':  [], 'test_acc':  [],
    'train_iou_per_class': [], 'test_iou_per_class': [],
}


def mask_to_color(mask: np.ndarray) -> np.ndarray:
    return CLASS_COLORS[np.clip(mask.astype(np.int64), 0, NUM_CLASSES - 1)]


def error_map(gt: np.ndarray, pred: np.ndarray) -> np.ndarray:
    out = np.zeros((gt.shape[0], gt.shape[1], 3), dtype=np.uint8)
    gt_fg, pr_fg = gt > 0, pred > 0
    out[gt_fg & pr_fg]    = [0, 200, 0]
    out[gt_fg & (~pr_fg)] = [255, 0, 0]
    out[(~gt_fg) & pr_fg] = [255, 165, 0]
    return out


def export_all_test(save_root: str, epoch: int):
    mask_dir = os.path.join(save_root, 'pred_mask')
    vis_dir  = os.path.join(save_root, 'pred_vis')
    os.makedirs(mask_dir, exist_ok=True)
    os.makedirs(vis_dir, exist_ok=True)

    model.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for img, label, name in tqdm(test_iter, desc=f'Export@E{epoch}', unit='img'):
            img, label = img.to(device), label.to(device)
            pred = model(img).argmax(dim=1)

            pred_np = pred[0].cpu().numpy().astype(np.uint8)
            gt_np   = label[0].cpu().numpy().astype(np.uint8)
            stem    = name[0]

            if SAVE_PRED_MASK_PNG:
                Image.fromarray(pred_np, mode='L').save(
                    os.path.join(mask_dir, f'{stem}.png'))

            if SAVE_PRED_VIS_PNG:
                x = img[0].cpu().numpy()
                wac = x[0] * CHANNEL_STD[0] + CHANNEL_MEAN[0]
                wac = np.clip(wac, 0, 1)

                fig, axes = plt.subplots(1, 4, figsize=(16, 4))
                axes[0].imshow(wac, cmap='gray');          axes[0].set_title(f'WAC - {stem}', fontsize=8)
                axes[1].imshow(mask_to_color(gt_np));      axes[1].set_title('GT')
                axes[2].imshow(mask_to_color(pred_np));    axes[2].set_title('Pred')
                axes[3].imshow(error_map(gt_np, pred_np)); axes[3].set_title('Error(G=TP,R=FN,O=FP)')
                for ax in axes:
                    ax.axis('off')
                plt.tight_layout()
                plt.savefig(os.path.join(vis_dir, f'{stem}.png'), dpi=120)
                plt.close(fig)

    print(f'已导出 {len(os.listdir(mask_dir))} 张 pred_mask + {len(os.listdir(vis_dir))} 张 pred_vis')


def plot_training_curves():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ep = history['epoch']

    axes[0, 0].plot(ep, history['train_loss'], 'o-', ms=3, label='Train')
    axes[0, 0].plot(ep, history['test_loss'],  's-', ms=3, label='Test')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curve'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(ep, history['train_miou'], 'o-', ms=3, label='Train')
    axes[0, 1].plot(ep, history['test_miou'],  's-', ms=3, label='Test')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('mIoU')
    axes[0, 1].set_title('mIoU Curve'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    arr = np.array(history['train_iou_per_class'])
    for c in range(NUM_CLASSES):
        axes[1, 0].plot(ep, arr[:, c], 'o-', ms=2, lw=1.5,
                        color=CLASS_COLORS_PLT[c], label=CLASS_NAMES[c])
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('IoU')
    axes[1, 0].set_title('Train Per-Class IoU'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    arr = np.array(history['test_iou_per_class'])
    for c in range(NUM_CLASSES):
        axes[1, 1].plot(ep, arr[:, c], 's-', ms=2, lw=1.5,
                        color=CLASS_COLORS_PLT[c], label=CLASS_NAMES[c])
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('IoU')
    axes[1, 1].set_title('Test Per-Class IoU'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'training_curves.png'), dpi=150)
    plt.show()
    print('Saved: training_curves.png')


csv_path = os.path.join(RESULT_DIR, 'epoch_metrics.csv')
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    header = ['epoch', 'train_loss', 'test_loss', 'train_miou', 'test_miou', 'train_acc', 'test_acc']
    for c in range(NUM_CLASSES):
        header += [f'train_iou_{c}', f'test_iou_{c}']
    w.writerow(header)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []
    hist = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.float64)
    optimizer.zero_grad()

    pbar = tqdm(train_iter, desc=f'Epoch {epoch}/{NUM_EPOCHS}', unit='batch')
    for step, (img, label, name) in enumerate(pbar):
        if MAX_STEPS > 0 and step >= MAX_STEPS:
            break
        img, label = img.to(device), label.to(device)

        with torch.amp.autocast('cuda'):
            logits = model(img)
            l = combined_loss(logits, label) / ACCUM_STEPS

        scaler.scale(l).backward()
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_iter):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_losses.append(l.item() * ACCUM_STEPS)
        with torch.no_grad():
            pred = logits.argmax(dim=1)
            hist += metrics.multiclass_confusion(pred, label, NUM_CLASSES).double()

        if step % 50 == 0:
            pbar.set_postfix(loss=f'{np.mean(train_losses[-50:]):.4f}')

    m = metrics.metrics_from_hist(hist)
    avg_loss = float(np.mean(train_losses))
    print(f'\n[Train] loss: {avg_loss:.4f}  mIoU: {m["miou"]:.4f}  acc: {m["accuracy"]:.4f}')
    print('  IoU:', ' '.join(f'{v:.4f}' for v in m['iou_per_class']))

    scheduler.step()

    model.eval()
    test_losses = []
    test_hist = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.float64)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for img, label, name in tqdm(test_iter, desc='Testing', unit='img'):
            if MAX_STEPS > 0 and len(test_losses) >= MAX_STEPS:
                break
            img, label = img.to(device), label.to(device)
            logits = model(img)
            test_losses.append(combined_loss(logits, label).item())
            pred = logits.argmax(dim=1)
            test_hist += metrics.multiclass_confusion(pred, label, NUM_CLASSES).double()

    tm = metrics.metrics_from_hist(test_hist)
    test_avg_loss = float(np.mean(test_losses))
    print(f'[Test]  loss: {test_avg_loss:.4f}  mIoU: {tm["miou"]:.4f}  acc: {tm["accuracy"]:.4f}')
    print('  IoU:', ' '.join(f'{v:.4f}' for v in tm['iou_per_class']))

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_loss)
    history['train_miou'].append(float(m['miou']))
    history['train_acc'].append(float(m['accuracy']))
    history['test_loss'].append(test_avg_loss)
    history['test_miou'].append(float(tm['miou']))
    history['test_acc'].append(float(tm['accuracy']))
    history['train_iou_per_class'].append([float(v) for v in m['iou_per_class']])
    history['test_iou_per_class'].append([float(v) for v in tm['iou_per_class']])

    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        row = [epoch, f'{avg_loss:.6f}', f'{test_avg_loss:.6f}',
               f'{float(m["miou"]):.6f}', f'{float(tm["miou"]):.6f}',
               f'{float(m["accuracy"]):.6f}', f'{float(tm["accuracy"]):.6f}']
        for c in range(NUM_CLASSES):
            row += [f'{m["iou_per_class"][c]:.6f}', f'{tm["iou_per_class"][c]:.6f}']
        csv.writer(f).writerow(row)

    with open(os.path.join(RESULT_DIR, 'history.json'), 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2, ensure_ascii=False)

    # 保存最优 + 导出
    if tm['miou'] > best_miou:
        best_miou = float(tm['miou'])
        no_improve_count = 0
        torch.save(model.state_dict(), os.path.join(RESULT_DIR, f'best_{MODEL_SIZE}.pth'))
        print(f'>>> Best model saved! mIoU={best_miou:.4f}')

        if SAVE_ALL_TEST_ON_BEST:
            if best_export_dir and os.path.isdir(best_export_dir):
                shutil.rmtree(best_export_dir, ignore_errors=True)
            best_export_dir = os.path.join(RESULT_DIR, f'best_epoch_{epoch:02d}_miou_{best_miou:.4f}')
            export_all_test(save_root=best_export_dir, epoch=epoch)
    else:
        no_improve_count += 1
        print(f'  No improvement ({no_improve_count}/{PATIENCE})')

    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_miou': best_miou,
        }, os.path.join(RESULT_DIR, f'ckpt_epoch{epoch}.pth'))
        print(f'Checkpoint saved at epoch {epoch}')

    # Early stopping
    if EARLY_STOP and no_improve_count >= PATIENCE:
        print(f'\n*** Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs) ***')
        break

print(f'\nTraining done! Best mIoU = {best_miou:.4f}')
print(f'Epoch metrics: {csv_path}')

plot_training_curves()

In [ ]:
# Cell 12: 打包结果 + 保存最终模型 + 自动关闭
import zipfile

torch.save(model.state_dict(), os.path.join(RESULT_DIR, f'final_{MODEL_SIZE}.pth'))
print('Final model saved!')

# 打包 result 为 zip 方便下载
zip_path = '/kaggle/working/result.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(RESULT_DIR):
        for f in files:
            fpath = os.path.join(root, f)
            arcname = os.path.relpath(fpath, '/kaggle/working')
            zf.write(fpath, arcname)
print(f'打包完成: {zip_path}')
!ls -lh /kaggle/working/result.zip
!ls -lh /kaggle/working/result/

# 自动关闭 session
import os
os.system('kill -9 -1')